# lora-eval-lab: the GPU steps
---

This notebook runs the only two steps that need a GPU: base generation, training, tuned generation.
No logic lives here; every cell calls the package. Runtime: T4 (Runtime > Change runtime type > T4 GPU).

Order matters and mirrors `PROCESS.md`: the untuned base model generates first (the control), then training, then the tuned model generates with the identical prompt and decoding.

## 1. Install and clone
---

In [ ]:
!pip install -q unsloth
!git clone -q https://github.com/J-Jurza/lora-eval-lab.git
%cd lora-eval-lab
!pip install -q -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Drive, for artefacts that outlive the session
---

Every stage copies its output here as soon as it finishes, so a pre-emption costs one stage.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/lora-eval-lab"
!mkdir -p $DRIVE/results $DRIVE/adapter

## 3. Data
---

Downloads the pinned CSVs and verifies their checksums. The held-out ids are already frozen in the repo.

In [ ]:
!python -m lora_eval_lab.data --download --stats

## 4. Base generations (the control)
---

Untuned Qwen2.5-1.5B-Instruct over the 199 held-out dialogues, greedy decoding. Resumable: rerun the cell if the session drops.

In [ ]:
!python -m lora_eval_lab.generate --tag base --batch-size 8
!cp results/generations_base.jsonl $DRIVE/results/

## 5. Train (step 3, filled in at that step)
---

In [ ]:
# !python -m lora_eval_lab.train --out adapter
# !cp -r adapter $DRIVE/adapter/ && cp results/train_config.json results/train_log.jsonl $DRIVE/results/

## 6. Tuned generations (step 4)
---

Same prompt, same decoding, base model plus the adapter.

In [ ]:
# !python -m lora_eval_lab.generate --tag tuned --adapter adapter --batch-size 8
# !cp results/generations_tuned.jsonl $DRIVE/results/

## 7. Take the results home
---

Download `results/*.jsonl` and commit them locally. Judging and metrics run on CPU.

In [ ]:
from google.colab import files
files.download("results/generations_base.jsonl")